# 03. Advanced Model Evaluation

Goal: compare quality improvements from model upgrade versus prompt-only changes.


## Step 1: Install Dependencies


In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r requirements.txt

## Step 2: Load Shared Utilities


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.student_api import process_addresses, process_batch_addresses, process_single_address

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Student Identity Parameter


In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ['WORKSHOP_EMAIL']

In [ ]:
# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')



## Step 4: Run Advanced Model Stage


In [ ]:
# Prompt is editable directly in the notebook (no external prompt files).
USE_CUSTOM_PROMPT = True
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Town Name should be city/locality only.
- Postal Code should include only postal token(s).
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage='advanced',
    model=os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro'),
    temperature=0.0,
    top_p=0.95,
    top_k=40,
    custom_prompt_template=custom_prompt,
    max_workers=8,
)
report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)

print(report['summary'])
report['field_metrics']


## Step 5: Save Advanced-Model Artifacts


In [ ]:
# Save predictions and evaluation files for stage comparison.
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_path = out_dir / '03_advanced_predictions.csv'
report_dir = out_dir / 'report_03_advanced'
pred_df.to_csv(pred_path, index=False)
save_evaluation_report(report, report_dir)

print('Saved predictions:', pred_path)
print('Saved report:', report_dir)


## Assignment

1. Compare baseline, prompt-tuned, and advanced results.
2. Summarize cost-quality tradeoffs.
3. Continue with `04_two_stage_with_kb.ipynb`.
